# GNN Physics Simulator — WCSPH Rollout Visualization

Generate a fresh WCSPH fluid trajectory (from the same distribution used for fine-tuning), run a full autoregressive rollout using the fine-tuned checkpoint, and compare predicted vs ground-truth particle positions side by side.

In [ ]:
import os
if(os.getcwd().split('/')[-1] == 'notebooks'):
  os.chdir('..')

import torch
import numpy as np
import yaml
import matplotlib.pyplot as plt
from matplotlib import animation
from IPython.display import HTML
from tqdm.notebook import tqdm

from models import GNNSimulator
from dataset import load_raw_data, build_graph
from simulation import FluidSimulation

torch.set_num_threads(12)

## 1. Load Config & Checkpoint

In [ ]:
CHECKPOINT_PATH = 'checkpoints/best_model_wcsph.pt'
CONFIG_PATH = 'configs/wcsph_transfer.yaml'

# Load config
with open(CONFIG_PATH, 'r') as f:
    config = yaml.safe_load(f)

# Load checkpoint
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
checkpoint = torch.load(CHECKPOINT_PATH, map_location=device, weights_only=False)

# Build model and load weights
model = GNNSimulator.from_config(config).to(device)
model.load_state_dict(checkpoint['model_state_dict'])

model.eval()

print(f"Loaded checkpoint from epoch {checkpoint['epoch']}, step {checkpoint['global_step']}")
print("Normalizer statistics reset for rollout derivation.")
print(f"Validation loss: {checkpoint['val_loss']:.6f}")
print(f"Device: {device}")

## 2. Generate a WCSPH Trajectory

Generate a trajectory from the same distribution used in `generate_fluid_dataset.py` for fine-tuning: randomized particle count (200–400), randomized drop position, standard WCSPH physics parameters.

In [ ]:
# ─── WaterDrop-aligned parameters (same as generate_fluid_dataset.py) ───
WATERDROP_BOUNDS = [[0.1, 0.9], [0.1, 0.9]]
WATERDROP_DT = 0.0025          # Effective frame-to-frame dt
INTEGRATION_DT = 0.0005        # Sub-step dt for WCSPH stability
SAVE_EVERY = 5                 # INTEGRATION_DT * SAVE_EVERY = WATERDROP_DT
SMOOTHING_LENGTH = 0.015       # = connectivity_radius
SEQUENCE_LENGTH = 1000         # Frames per trajectory
POSITION_SCALE = 0.8           # Container size: [0, 0.8], shifted by +0.1 → [0.1, 0.9]
OFFSET = 0.1                   # Coordinate shift to align with WaterDrop bounds
PARTICLE_TYPE_FLUID = 5        # DeepMind's encoding for fluid particles

# Random seed for this visualization (change for different trajectories)
VIZ_SEED = 123
rng = np.random.RandomState(VIZ_SEED)

# Random particle count in [200, 400]
num_particles = rng.randint(2000, 4001)


# Total simulation time to produce SEQUENCE_LENGTH + 1 frames
total_time = SEQUENCE_LENGTH * SAVE_EVERY * INTEGRATION_DT

sim = FluidSimulation(
    num_particles=num_particles,
    gravitational_constant=9.81,
    softening_length=SMOOTHING_LENGTH,
    integrator='symplectic_euler',
    position_scale=POSITION_SCALE,
    rest_density=1000.0,
    stiffness=1000.0,
    exponent=7.0,
    viscosity=0.3,
)

# Randomized initial drop center within margins
margin = 0.15 * POSITION_SCALE
cx = rng.uniform(margin, POSITION_SCALE - margin)
cy = rng.uniform(0.3 * POSITION_SCALE, POSITION_SCALE - margin)

sim.initialize_random_state(
    position_scale=POSITION_SCALE,
    velocity_scale=0.0,
    mass_range=(1.0, 1.0),
    start=(cx, cy),
)

print(f"Generating trajectory: {num_particles} particles, drop center=({cx:.3f}, {cy:.3f})")
positions, _, _ = sim.simulate(
    total_time=total_time,
    dt=INTEGRATION_DT,
    save_every=SAVE_EVERY,
    quiet=False
)

# Trim and shift to WaterDrop coordinate system
positions = (positions[:SEQUENCE_LENGTH + 1] + OFFSET).astype(np.float32)

# Derive ground truth velocities as spatial frame deltas to match DeepMind format
gt_velocities = (positions[1:] - positions[:-1]).astype(np.float32)
gt_positions = positions[1:].astype(np.float32)

particle_type = np.full(num_particles, PARTICLE_TYPE_FLUID, dtype=np.int64)
mass = np.ones(num_particles, dtype=np.float32)

T, N, D = gt_positions.shape
print(f"Trajectory: {T} timesteps, {N} particles, {D}D")
print(f"Position range: [{gt_positions.min():.4f}, {gt_positions.max():.4f}]")

# Build metadata dict matching WaterDrop format
metadata = {
    'bounds': WATERDROP_BOUNDS,
    'default_connectivity_radius': SMOOTHING_LENGTH,
    'dt': WATERDROP_DT,
}
bounds = metadata['bounds']

## 3. Autoregressive Rollout

Starting from the initial state (first `history_window + 1` frames), predict accelerations one step at a time and integrate forward to produce the full predicted trajectory.

In [ ]:
data_cfg = config['data']
history_window = data_cfg.get('history_window', 5)
connectivity_radius = metadata.get('default_connectivity_radius', 0.015)

# Initialize from ground truth (keep on CPU for plotting later)
pred_positions = np.zeros_like(gt_positions)  # [T, N, D]
pred_positions[:history_window + 1] = gt_positions[:history_window + 1].copy()

# Move invariant properties to device
particle_type_t = torch.from_numpy(particle_type).to(device)
mass_t = torch.from_numpy(mass).to(device)
dummy_acc_t = torch.zeros(N, D, dtype=torch.float32, device=device)

# Velocity buffer: store the last `history_window` velocities directly on GPU
vel_history_t = torch.from_numpy(gt_velocities[:history_window].astype(np.float32)).to(device)  # [C, N, D]

# Current position and velocity directly on GPU
current_pos_t = torch.from_numpy(gt_positions[history_window].astype(np.float32)).to(device)  # [N, D]
current_vel_t = torch.from_numpy(gt_velocities[history_window - 1].astype(np.float32)).to(device)  # [N, D]

with torch.inference_mode():
    for t in tqdm(range(history_window, T - 1), desc='Rollout'):
        # Flatten velocity history [C, N, D] -> [N, C*D] on GPU
        vel_flat_t = vel_history_t.transpose(0, 1).reshape(N, history_window * D)
        
        # Build graph from current state (no CPU transfer needed! radius_graph executes fast on GPU)
        graph = build_graph(
            positions=current_pos_t,
            velocities=vel_flat_t,
            particle_type=particle_type_t,
            masses=mass_t,
            target_acc=dummy_acc_t,
            connectivity_radius=connectivity_radius,
            bounds=bounds
        )
        
        # Predict acceleration directly on GPU
        predicted_acc_t = model(graph)  # [N, D]
        
        # Semi-implicit Euler integration on GPU: vel += acc, pos += vel
        new_vel_t = current_vel_t + predicted_acc_t
        new_pos_t = current_pos_t + new_vel_t
        
        # Store predicted position back to CPU buffer for plotting
        pred_positions[t + 1] = new_pos_t.cpu().numpy()
        
        # Shift velocity history window via pure tensor concatenation
        vel_history_t = torch.cat([vel_history_t[1:], new_vel_t.unsqueeze(0)], dim=0)
        current_pos_t = new_pos_t
        current_vel_t = new_vel_t

print("Rollout complete!")

## 4. Compute Rollout Error

In [ ]:
# Per-step position MSE
rollout_start = history_window + 1
step_mse = np.mean((pred_positions[rollout_start:] - gt_positions[rollout_start:])**2, axis=(1, 2))

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(range(rollout_start, T), step_mse, color='#e74c3c', linewidth=1.5)
ax.set_xlabel('Timestep')
ax.set_ylabel('Position MSE')
ax.set_title('WCSPH Rollout Error Over Time')
ax.set_yscale('log')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Mean rollout MSE: {step_mse.mean():.6f}")
print(f"Final step MSE: {step_mse[-1]:.6f}")

## 5. Side-by-Side Animation

In [ ]:
# Determine plot bounds from ground truth
if bounds is not None:
    x_lim = bounds[0]
    y_lim = bounds[1]
else:
    margin = 0.1
    x_lim = [gt_positions[:, :, 0].min() - margin, gt_positions[:, :, 0].max() + margin]
    y_lim = [gt_positions[:, :, 1].min() - margin, gt_positions[:, :, 1].max() + margin]

# Animation at 60fps with accurate simulation time
TARGET_FPS = 60
sim_dt = metadata.get('dt', 0.0025)  # effective frame-to-frame dt
sim_fps = 1.0 / sim_dt               # simulation frames per second of real time
frame_skip = max(1, int(sim_fps / TARGET_FPS))  # skip to match real-time playback
frames = list(range(0, T, frame_skip))
interval_ms = 1000.0 / TARGET_FPS    # ~16.67ms per frame

print(f"Animation: {frame_skip} skip")

fig, (ax_gt, ax_pred) = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle('WCSPH Rollout: Ground Truth vs GNN Prediction', fontsize=14, fontweight='bold')

for ax, title in [(ax_gt, 'Ground Truth (WCSPH)'), (ax_pred, 'GNN Prediction')]:
    ax.set_xlim(x_lim)
    ax.set_ylim(y_lim)
    ax.set_aspect('equal')
    ax.set_title(title)
    ax.set_facecolor('#ffffff')

from matplotlib.colors import LinearSegmentedColormap
water_colormap = LinearSegmentedColormap.from_list('ocean_blue', ['#0088cc', '#000033'])
water_colormap_gnn = LinearSegmentedColormap.from_list('ocean_purple', ['#e175ff', '#5a0778'])
particle_colors = np.arange(N)

scatter_gt = ax_gt.scatter(gt_positions[0, :, 0], gt_positions[0, :, 1], s=4, c=particle_colors, cmap=water_colormap, alpha=0.9)
scatter_pred = ax_pred.scatter(pred_positions[0, :, 0], pred_positions[0, :, 1], s=4, c=particle_colors, cmap=water_colormap_gnn, alpha=0.9)
time_text = fig.text(0.5, 0.02, '', ha='center', fontsize=11)

def init():
    scatter_gt.set_offsets(gt_positions[0, :, :2])
    scatter_pred.set_offsets(pred_positions[0, :, :2])
    return scatter_gt, scatter_pred, time_text

def animate(frame_idx):
    t = frames[frame_idx]
    scatter_gt.set_offsets(gt_positions[t, :, :2])
    scatter_pred.set_offsets(pred_positions[t, :, :2])
    time_text.set_text(f't = {t * sim_dt:.3f}s  (step {t}/{T-1})')
    return scatter_gt, scatter_pred, time_text

anim = animation.FuncAnimation(fig, animate, init_func=init,
                               frames=len(frames), interval=interval_ms, blit=True)
plt.close(fig)
HTML(anim.to_html5_video())